# Getting Started with MVTracker
### Community Contribution by [Dante Boeri](https://github.com/DronteBronte)

Welcome to the hands-on interactive tutorial for **MVTracker** (Multi-View Tracker).

## What is MVTracker?

In the field of visual computing, **Point Tracking** (Tracking Any Point, or TAP) is the task of estimating the trajectory of a specific physical point across a video sequence. Traditional methods (like Optical Flow, RAFT, or CoTracker) operate entirely in **2D image space**: they track pixels from frame to frame, which struggles with severe occlusions or when objects rotate out of view.

**MVTracker** elevates this problem to **3D space**. Given multiple synchronized camera views and depth information, it "lifts" the 2D images into a unified 3D point cloud. It then tracks the physical 3D coordinates of points over time using a Spatio-Temporal Transformer. By reasoning directly in 3D, it naturally handles occlusions and multi-camera consistency.

## What this notebook covers

We will walk through the **entire pipeline** step by step, using **sequence 0 of the DexYCB dataset**: a multi-view RGBD recording of a hand manipulating YCB objects, complete with ground-truth 3D trajectories:

1. **Environment Setup** — Install all dependencies.
2. **Load Model & Data** — Load the pretrained model and DexYCB sequence 0.
3. **Understand the Inputs** — Inspect every input tensor (RGBs, depths, intrinsics, extrinsics, query points) and visualize them.
4. **3D Lifting (Unprojection)** — Manually perform the pixel → camera → world coordinate transform to understand how MVTracker builds its 3D scene representation.
5. **Run Inference** — Execute the full tracking pipeline.
6. **Understand the Outputs** — Inspect the predicted 3D trajectories and visibilities.
7. **Visualize Trajectories** — Plot the 3D tracks.
7. **Visualize Trajectories** — Plot their 2D reprojections onto image frames.
9. **Evaluate with Metrics** — Compute the standard TAP-Vid evaluation metrics (Jaccard, Points Within Threshold, Survival, Occlusion Accuracy) against **real ground truth** and learn what each number means.
10. **Export & Visualize in 3D** — Save results and generate a Rerun `.rrd` file for interactive 3D visualization.

Note: This notebook is designed to run **from a local clone** of the [ethz-vlg/mvtracker](https://github.com/ethz-vlg/mvtracker) repository. It was validated on **Python 3.10.12**, **PyTorch 2.3.0**, and **CUDA 12.1** (the same stack documented in the repository README).

## Step 1: Install Dependencies

This notebook assumes you have already set up the validated environment as described in the repo README:

```bash
conda create -n 3dpt python=3.10.12 -y
conda activate 3dpt
conda install pytorch==2.3.0 torchvision==0.18.0 torchaudio==2.3.0 pytorch-cuda=12.1 -c pytorch -c nvidia -y
pip install -r requirements.txt
```

The cell below installs the repo's **pinned minimal requirements** (if not already installed) and verifies the environment.

In [ ]:
import sys, subprocess

# Prevent Python from creating __pycache__ folders
sys.dont_write_bytecode = True

# Install the repo's pinned minimal requirements (idempotent).
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])

# Sanity-check the critical packages
import torch, numpy as np
print(f"Python:  {sys.version}")
print(f"PyTorch: {torch.__version__}  (CUDA available: {torch.cuda.is_available()})")
print(f"NumPy:   {np.__version__}")
print("All dependencies installed.")

## Step 2: Load MVTracker & Download DexYCB Sequence 0

MVTracker is loaded from the **local repository** via PyTorch Hub (`source="local"`). Under the hood (see `hubconf.py`), this:
1. Instantiates the `MVTracker` neural network (a CNN feature encoder + Spatio-Temporal Transformer).
2. Wraps it in an `EvaluationPredictor` that handles input resizing, query point batching, and sliding-window inference.
3. Downloads and loads the pretrained weights from Hugging Face.

We also download **sequence 0** of the **DexYCB** multi-view dataset, which is a short multi-view RGBD recording of a hand manipulating YCB objects.

In [ ]:
import contextlib
import io
import logging
import os
import tarfile
import warnings

import matplotlib.pyplot as plt
import numpy as np
import torch
from huggingface_hub import hf_hub_download

# Keep model loading quiet when optional video dependencies emit import-time noise.
os.environ.setdefault("PYGAME_HIDE_SUPPORT_PROMPT", "1")
os.environ.setdefault("SDL_AUDIODRIVER", "dummy")
os.environ.setdefault("XDG_RUNTIME_DIR", "/tmp")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# ── Load pretrained MVTracker from the LOCAL repo checkout ──
previous_logging_disable = logging.root.manager.disable
with warnings.catch_warnings(), contextlib.redirect_stderr(io.StringIO()):
    warnings.filterwarnings("ignore", category=SyntaxWarning, module=r"moviepy(\\.|$)")
    logging.disable(logging.CRITICAL)
    try:
        mvtracker = torch.hub.load(
            ".",
            "mvtracker",
            source="local",
            pretrained=True,
            device=device,
        )
    finally:
        logging.disable(previous_logging_disable)
print("MVTracker loaded successfully.\n")

# ── Download & extract DexYCB sequence 0 from Hugging Face ──
dexycb_root = "datasets/dex-ycb-multiview"
seq_0_name = "20200709-subject-01__20200709_141754"
seq_0_dir = os.path.join(dexycb_root, seq_0_name)

if os.path.isdir(seq_0_dir):
    print(f"DexYCB sequence 0 already present at: {seq_0_dir}")
else:
    print("Downloading DexYCB dataset from Hugging Face...")
    tar_path = hf_hub_download(
        repo_id="ethz-vlg/mv3dpt-datasets",
        filename="dex-ycb-multiview.tar.gz",
        repo_type="dataset",
        token=os.getenv("HF_TOKEN"),
    )
    print("Extracting sequence 0 only...")
    seq_0_prefix = f"dex-ycb-multiview/{seq_0_name}"
    os.makedirs(dexycb_root, exist_ok=True)
    with tarfile.open(tar_path) as tar:
        members = [m for m in tar.getmembers() if m.name.startswith(seq_0_prefix)]
        tar.extractall("datasets", members=members)
    print(f"Extracted to: {seq_0_dir}")

# ── Load sequence 0 via DexYCBMultiViewDataset ──
from mvtracker.datasets.dexycb_multiview_dataset import DexYCBMultiViewDataset

dataset = DexYCBMultiViewDataset(
    data_root=dexycb_root,
    max_videos=1,
    views_to_return=[0, 1, 2, 3],
    traj_per_sample=384,
    seed=72,
)
sample, _ = dataset[0]

rgbs         = sample.video.float()              # [V, T, 3, H, W]
depths       = sample.videodepth.float()          # [V, T, 1, H, W]
intrs        = sample.intrs.float()               # [V, T, 3, 3]
extrs        = sample.extrs.float()               # [V, T, 3, 4]
query_points = sample.query_points_3d.float()     # [N, 4] = (t, x, y, z)

# Ground truth (DexYCB ships with these!)
gt_tracks_3d      = sample.trajectory_3d.float()         # [T, N, 3] — world-space
gt_visibility     = sample.visibility                     # [V, T, N] — per-view
gt_category       = sample.trajectory_category             # [N] — 0=hand, 1=dynamic YCB, 2=static YCB
upscale_factor    = sample.track_upscaling_factor          # 1/6 — converts normalized → metric

V, T, _, H, W = rgbs.shape
N = query_points.shape[0]

# ── Assertions: fail fast if shapes are wrong ──
assert rgbs.ndim == 5 and rgbs.shape[2] == 3, f"Expected rgbs [V,T,3,H,W], got {rgbs.shape}"
assert depths.ndim == 5 and depths.shape[2] == 1, f"Expected depths [V,T,1,H,W], got {depths.shape}"
assert intrs.shape[-2:] == (3, 3), f"Expected intrs [...,3,3], got {intrs.shape}"
assert extrs.shape[-2:] == (3, 4), f"Expected extrs [...,3,4], got {extrs.shape}"
assert query_points.ndim == 2 and query_points.shape[1] == 4, f"Expected query_points [N,4], got {query_points.shape}"

print()
print("=== Loaded DexYCB Sequence 0 ===")
print(f"RGBs:         {rgbs.shape}  ->  [Views, Time, Channels, Height, Width]")
print(f"Depths:       {depths.shape}  ->  [Views, Time, 1, Height, Width]")
print(f"Intrinsics:   {intrs.shape}  ->  [Views, Time, 3, 3]")
print(f"Extrinsics:   {extrs.shape}  ->  [Views, Time, 3, 4]")
print(f"Query points: {query_points.shape}  ->  [Num_Points, 4]  (t, x, y, z)")
print(f"\n=> {V} camera views, {T} time frames, {H}x{W} resolution, {N} query points")

## Step 3: Deep Dive into Each Input

To track points in 3D, MVTracker requires a full **multi-view geometry** of the scene. Let's examine each input tensor and understand what it represents.

### 3.1 — RGB Images (`rgbs`)
The raw video frames from multiple synchronized cameras. Each pixel value is in `[0, 255]`. The model's CNN feature encoder will process these to extract dense visual features.

Shape: `[Views, Time, 3, Height, Width]`

### 3.2 — Depth Maps (`depths`)
For each pixel, the depth map stores the distance from the camera to the scene surface (in meters). A value of `0` means invalid/no depth.

Shape: `[Views, Time, 1, Height, Width]`

### 3.3 — Camera Intrinsics (`intrs`)
The **intrinsic matrix** $K$ is a $3 \times 3$ upper-triangular matrix encoding the internal properties of each camera:

$$K = \begin{bmatrix} f_x & 0 & c_x \\ 0 & f_y & c_y \\ 0 & 0 & 1 \end{bmatrix}$$

where $f_x, f_y$ are focal lengths (in pixels) and $(c_x, c_y)$ is the principal point (optical center). This matrix maps 3D camera-space points to 2D pixel coordinates.

### 3.4 — Camera Extrinsics (`extrs`)
The **extrinsic matrix** $[R|t]$ is a $3 \times 4$ matrix encoding the camera's pose in the world:

$$[R|t] = \begin{bmatrix} r_{11} & r_{12} & r_{13} & t_x \\ r_{21} & r_{22} & r_{23} & t_y \\ r_{31} & r_{32} & r_{33} & t_z \end{bmatrix}$$

where $R$ is a $3 \times 3$ rotation matrix and $t$ is a $3 \times 1$ translation vector. This transforms points from **world space** to **camera space**: $\mathbf{p}_{cam} = R \cdot \mathbf{p}_{world} + t$.

### 3.5 — Query Points (`query_points`)
Each query point is a 4-element vector $(t, x, y, z)$:
- $t$: the starting time frame index (the frame in which the point first appears)
- $(x, y, z)$: the initial 3D world-space coordinates of the point to track

The model's task is: **given this starting 3D location at time $t$, where does this physical point move in every subsequent frame?**

In [ ]:
# ── 3.1 Visualize RGB frames from each view at t=0 ──
fig, axes = plt.subplots(1, V, figsize=(5 * V, 5))
if V == 1:
    axes = [axes]
for v in range(V):
    img = rgbs[v, 0].permute(1, 2, 0).numpy().astype(np.uint8)  # [H, W, 3]
    axes[v].imshow(img)
    axes[v].set_title(f"View {v}, t=0", fontsize=14)
    axes[v].axis("off")
plt.suptitle("RGB Frames at t=0 from All Camera Views", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

# ── 3.2 Visualize depth maps ──
fig, axes = plt.subplots(1, V, figsize=(5 * V, 5))
if V == 1:
    axes = [axes]
for v in range(V):
    d = depths[v, 0, 0].numpy()  # [H, W]
    im = axes[v].imshow(d, cmap="turbo")
    axes[v].set_title(f"Depth View {v}, t=0", fontsize=14)
    axes[v].axis("off")
    plt.colorbar(im, ax=axes[v], fraction=0.046, pad=0.04, label="meters")
plt.suptitle("Depth Maps at t=0 (Turbo colormap)", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

# ── 3.3 Inspect intrinsics ──
print("=== Camera Intrinsics (View 0, t=0) ===")
print()
K0 = intrs[0, 0]
print(K0.numpy().round(2))
print()
print(f"Focal length: fx={K0[0,0]:.1f} px, fy={K0[1,1]:.1f} px")
print(f"Principal pt: cx={K0[0,2]:.1f} px, cy={K0[1,2]:.1f} px")

# ── 3.4 Inspect extrinsics ──
print("\n=== Camera Extrinsics (View 0, t=0) ===")
print()
Rt0 = extrs[0, 0]
print(Rt0.numpy().round(4))
R0 = Rt0[:3, :3]
t_vec0 = Rt0[:3, 3]
print()
print(f"Rotation (3x3):\n\n{R0.numpy().round(4)}")
print()
print(f"Translation: {t_vec0.numpy().round(4)}")

# Camera position in world space: C = -R^T @ t
cam_pos = -R0.T @ t_vec0
print(f"Camera position in world: {cam_pos.numpy().round(3)}")

# ── 3.5 Inspect query points ──
print(f"\n=== Query Points (first 10 of {N}) ===")
print(f"{'Idx':>4}  {'t':>4}  {'X':>8}  {'Y':>8}  {'Z':>8}")
for i in range(min(10, N)):
    qp = query_points[i]
    print(f"{i:4d}  {qp[0]:4.0f}  {qp[1]:8.3f}  {qp[2]:8.3f}  {qp[3]:8.3f}")

## Step 4: 3D Lifting (Unprojection)

The model fuses the 2D feature maps, depth maps, and camera parameters into a **single unified 3D point cloud** of the entire scene.

**The math** (for every pixel $(u, v)$ with depth $d$):

1. **Pixel → Camera Space** using the inverse intrinsic matrix $K^{-1}$:
$$\mathbf{p}_{cam} = d \cdot K^{-1} \begin{bmatrix} u \\ v \\ 1 \end{bmatrix}$$

2. **Camera → World Space** using the inverse extrinsic matrix $[R|t]^{-1}$:
$$\mathbf{p}_{world} = R^{-1}(\mathbf{p}_{cam} - t)$$

In the code (`init_pointcloud_from_rgbd` in `model_utils.py`), this is done in parallel across **all views and all pixels** using `torch.einsum`. The result is a massive point cloud where each 3D point carries its visual feature vector.

Let's do this manually to see it work!

In [ ]:
# ══════════════════════════════════════════════════════════
# Manual 3D Unprojection — Understanding the Geometry
# ══════════════════════════════════════════════════════════
# We'll unproject the depth maps from all views at t=0 into world-space 3D points.
# This is exactly what MVTracker does internally in `init_pointcloud_from_rgbd`.

from mvtracker.utils.basic import to_homogeneous, from_homogeneous

view_idx, time_idx = 0, 0  # Let's start with view 0, time 0

# Step 0: Get the camera parameters for this view
K      = intrs[view_idx, time_idx]    # [3, 3]
K_inv  = torch.inverse(K.float())     # [3, 3]
Rt     = extrs[view_idx, time_idx]    # [3, 4]
# Build the full 4x4 extrinsic matrix and invert it
Rt4x4  = torch.eye(4)
Rt4x4[:3, :] = Rt
Rt4x4_inv = torch.inverse(Rt4x4.float())

depth_map = depths[view_idx, time_idx, 0]  # [H, W]
rgb_img   = rgbs[view_idx, time_idx]       # [3, H, W]

# Step 1: Create a grid of pixel coordinates (u, v)
grid_v, grid_u = torch.meshgrid(
    torch.arange(H, dtype=torch.float32),
    torch.arange(W, dtype=torch.float32),
    indexing="ij",
)
# pixel_coords_homo shape: [H, W, 3] = (u, v, 1)
pixel_coords_homo = torch.stack([grid_u, grid_v, torch.ones_like(grid_u)], dim=-1)

# Step 2: Pixel → Camera Space: p_cam = depth * K_inv @ (u, v, 1)
camera_xyz = torch.einsum("ij,HWj->HWi", K_inv, pixel_coords_homo)
camera_xyz = camera_xyz * depth_map.unsqueeze(-1)  # Scale by depth

# Step 3: Camera → World Space: p_world = Rt4x4_inv @ (p_cam, 1)
camera_xyz_homo = to_homogeneous(camera_xyz)  # [H, W, 4]
world_xyz_homo  = torch.einsum("ij,HWj->HWi", Rt4x4_inv, camera_xyz_homo)
world_xyz       = from_homogeneous(world_xyz_homo)  # [H, W, 3]

# Filter out invalid depth (depth == 0)
valid_mask = depth_map > 0
pts_world  = world_xyz[valid_mask].numpy()     # [M, 3]
pts_color  = rgb_img.permute(1, 2, 0)[valid_mask].numpy().astype(np.uint8)  # [M, 3]

print(f"Unprojected {pts_world.shape[0]:,} valid 3D points from View {view_idx}, t={time_idx}")

# ── Visualize the 3D point cloud in a single 3D plot ──
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection="3d")

# Subsample for performance
subsample = np.random.choice(len(pts_world), min(20000, len(pts_world)), replace=False)
ax.scatter(
    pts_world[subsample, 0], pts_world[subsample, 1], pts_world[subsample, 2],
    c=pts_color[subsample] / 255.0, s=0.5, alpha=0.8,
)

# Plot query points on top
qp = query_points[:, 1:].numpy()
ax.scatter(qp[:, 0], qp[:, 1], qp[:, 2], c="red", s=30, marker="x", label="Query Points", zorder=5)

ax.set_xlabel("X (world)")
ax.set_ylabel("Y (world)")
ax.set_zlabel("Z (world)")
ax.set_title(f"3D Point Cloud from View {view_idx}, t={time_idx}")
ax.legend()
ax.view_init(elev=30, azim=45)

margin = 2.0
ax.set_xlim(qp[:, 0].min() - margin, qp[:, 0].max() + margin)
ax.set_ylim(qp[:, 1].min() - margin, qp[:, 1].max() + margin)
ax.set_zlim(qp[:, 2].min() - margin, qp[:, 2].max() + margin)

plt.tight_layout()
plt.show()

## Step 5: Run Inference

Now we run the full MVTracker forward pass. Under the hood, the following happens:

1. **Feature Extraction**: The RGB frames are normalized to `[−1, 1]` and passed through a CNN (`BasicEncoder`) to produce 128-dimensional feature maps at stride 4.
2. **3D Lifting**: For each pyramid level, the code calls `init_pointcloud_from_rgbd` to create a 3D point cloud from the feature maps and depth maps. That's what we did manually above, but now with learned features attached.
3. **Sliding Window**: MVTracker processes the sequence in overlapping windows of 12 frames (configurable via `sliding_window_len`). Adjacent windows overlap by 50%.
4. **Iterative Refinement** (4 iterations per window):
   - **Correlation Sampling**: Using KNN, find the K nearest 3D neighbors to each tracked point and compute feature correlations.
   - **Transform → Update**: The Spatio-Temporal Transformer produces a coordinate delta (`d_coord`) and a feature delta (`d_feats`). These are added to the running estimate.
5. **Visibility Prediction**: After the final iteration, a linear head predicts visibility logits from the point features.

In [ ]:
torch.set_float32_matmul_precision("high")
amp_dtype = torch.bfloat16 if (device == "cuda" and torch.cuda.get_device_capability()[0] >= 8) else torch.float16

print("Running MVTracker forward pass...")
print(f"  Input: {V} views x {T} frames x {H}x{W} px,  {N} query points")
print(f"  AMP dtype: {amp_dtype}")

with torch.no_grad(), torch.cuda.amp.autocast(enabled=device == "cuda", dtype=amp_dtype):
    results = mvtracker(
        rgbs=rgbs[None].to(device) / 255.0,          # [1, V, T, 3, H, W]
        depths=depths[None].to(device),                # [1, V, T, 1, H, W]
        intrs=intrs[None].to(device),                  # [1, V, T, 3, 3]
        extrs=extrs[None].to(device),                  # [1, V, T, 3, 4]
        query_points_3d=query_points[None].to(device), # [1, N, 4]
    )

# Extract outputs — squeeze batch dim (B=1 always for inference)
pred_tracks   = results["traj_e"][0].cpu()      # [T, N, 3] — 3D world-space trajectories
pred_vis      = results["vis_e"][0].cpu()        # [T, N]    — boolean visibility mask
pred_vis_prob = results["vis_e_as_prob"][0].cpu() # [T, N]   — visibility probabilities

print()
print(f"pred_tracks shape: {pred_tracks.shape}  (expected [{T}, {N}, 3])")
print(f"pred_vis shape:    {pred_vis.shape}  (expected [{T}, {N}])")

# Assertions
assert pred_tracks.ndim == 3 and pred_tracks.shape[-1] == 3, f"Unexpected traj_e shape: {pred_tracks.shape}"
assert pred_vis.ndim == 2, f"Unexpected vis_e shape: {pred_vis.shape}"
assert pred_tracks.shape[:2] == pred_vis.shape, "traj_e and vis_e time/point dims must match"
print("\nInference complete. All output shape assertions passed.")

## Step 6: Understanding the Outputs

The model returns a dictionary. The key outputs are:

1. **`traj_e`** — `[Time, Num_Points, 3]`: The predicted 3D trajectories in **world space**. For every requested query point, this contains its estimated $(x, y, z)$ coordinates at every time step.

   - **How to read it**: `pred_tracks[t, i, :]` gives the 3D position of point $i$ at frame $t$.

2. **`vis_e`** — `[Time, Num_Points]`: A **boolean visibility mask** (thresholded internally at 0.5).
   - `True` means the model is confident the point is visible and its trajectory is reliable.
   - `False` means the point is likely occluded, out of the camera's field of view, or lost.

3. **`vis_e_as_prob`** — `[Time, Num_Points]`: The raw visibility **probabilities** (float, 0–1) before thresholding.

Let's inspect them.

In [ ]:
# ── Inspect the outputs ──
T_out, N_out, _ = pred_tracks.shape

print("=== MVTracker Outputs ===")
print(f"  Trajectories (traj_e):      {pred_tracks.shape}  ->  [Time={T_out}, Points={N_out}, XYZ=3]")
print(f"  Visibility   (vis_e):       {pred_vis.shape}  ->  [Time={T_out}, Points={N_out}]  (boolean)")
print(f"  Visibility   (vis_e_prob):  {pred_vis_prob.shape}  ->  [Time={T_out}, Points={N_out}]  (float)")

print(f"\n=== Coordinate Statistics (in world-space) ===")
print(f"  X range: [{pred_tracks[..., 0].min():.3f}, {pred_tracks[..., 0].max():.3f}]")
print(f"  Y range: [{pred_tracks[..., 1].min():.3f}, {pred_tracks[..., 1].max():.3f}]")
print(f"  Z range: [{pred_tracks[..., 2].min():.3f}, {pred_tracks[..., 2].max():.3f}]")

print(f"\n=== Visibility Statistics ===")
print(f"  Fraction visible: {pred_vis.float().mean():.3f}")
print(f"  Probability range: [{pred_vis_prob.min():.3f}, {pred_vis_prob.max():.3f}]")

# ── Show a few trajectories ──
print(f"\n=== Example: Trajectory of Point 100 over first 10 frames ===")
print(f"{'Frame':>6}  {'X':>8}  {'Y':>8}  {'Z':>8}  {'Vis':>6}")
for t in range(min(10, T_out)):
    pos = pred_tracks[t, 100]
    v = "YES" if pred_vis[t, 100] else "no"
    print(f"{t:6d}  {pos[0]:8.3f}  {pos[1]:8.3f}  {pos[2]:8.3f}  {v:>6}")

## Step 7: Visualize 3D Trajectories

Let's plot the predicted 3D trajectories. Each colored line represents the path of a single physical point through 3D space over time.

In [ ]:
# ── Plot 3D trajectories ──
n_tracks_to_plot = min(50, N_out)
cmap = plt.get_cmap("hsv", n_tracks_to_plot)

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection="3d")

for i in range(n_tracks_to_plot):
    traj = pred_tracks[:, i].numpy()  # [T, 3]
    color = cmap(i)
    ax.plot(traj[:, 0], traj[:, 1], traj[:, 2], c=color, alpha=0.6, linewidth=0.8)
    # Start point (circle)
    ax.scatter(
        traj[0, 0], traj[0, 1], traj[0, 2],
        c=[color], s=25, marker="o", zorder=5, edgecolors="k", linewidths=0.3,
        label="Start" if i == 0 else "",
    )
    # End point (X)
    ax.scatter(
        traj[-1, 0], traj[-1, 1], traj[-1, 2],
        c=[color], s=30, marker="X", zorder=5, edgecolors="k", linewidths=0.3,
        label="End" if i == 0 else "",
    )

ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")
ax.set_title(f"Predicted 3D Trajectories ({n_tracks_to_plot} tracks)")
ax.view_init(elev=30, azim=45)

all_trajs = pred_tracks[:, :n_tracks_to_plot].numpy()  # [T, n, 3]
margin = 0.5
ax.set_xlim(all_trajs[..., 0].min() - margin, all_trajs[..., 0].max() + margin)
ax.set_ylim(all_trajs[..., 1].min() - margin, all_trajs[..., 1].max() + margin)
ax.set_zlim(all_trajs[..., 2].min() - margin, all_trajs[..., 2].max() + margin)

ax.legend()
plt.tight_layout()
plt.show()

## Step 8: Reproject 3D Tracks onto 2D Images

A powerful way to verify the 3D tracking is to **project the 3D trajectories back onto the 2D camera images**.

The math is the reverse of the unprojection:
1. **World → Camera**: $\mathbf{p}_{cam} = R \cdot \mathbf{p}_{world} + t$
2. **Camera → Pixel**: $\mathbf{p}_{pixel} = K \cdot \mathbf{p}_{cam}$, then divide by the $z$ component.

This is done by `world_space_to_pixel_xy_and_camera_z` in the codebase. If the 3D tracking is correct, the projected dots should land on the correct objects in every camera view.

In [ ]:
from mvtracker.models.core.model_utils import world_space_to_pixel_xy_and_camera_z

# ── Project 3D tracks onto each camera view ──
# pred_tracks is [T, N, 3]

n_tracks_to_show = min(30, N_out)
cmap = plt.get_cmap("hsv", n_tracks_to_show)

# Pick 3 time frames to visualize
show_frames = [0, T_out // 2, T_out - 1]

for v in range(V):
    fig, axes = plt.subplots(1, len(show_frames), figsize=(6 * len(show_frames), 5))
    if len(show_frames) == 1:
        axes = [axes]

    # Project 3D world points to this camera's 2D pixel coordinates
    pixel_xy, camera_z = world_space_to_pixel_xy_and_camera_z(
        world_xyz=pred_tracks,     # [T, N, 3]
        intrs=intrs[v],            # [T, 3, 3]
        extrs=extrs[v],            # [T, 3, 4]
    )
    pixel_xy = pixel_xy.numpy()    # [T, N, 2]
    camera_z = camera_z.numpy()    # [T, N, 1]

    for ax_idx, t in enumerate(show_frames):
        img = rgbs[v, t].permute(1, 2, 0).numpy().astype(np.uint8)
        axes[ax_idx].imshow(img)

        for i in range(n_tracks_to_show):
            # Only plot if the point is in front of the camera (z > 0) and visible
            if camera_z[t, i, 0] > 0 and pred_vis[t, i]:
                px, py = pixel_xy[t, i]
                if 0 <= px < W and 0 <= py < H:
                    axes[ax_idx].scatter(
                        px, py, c=[cmap(i)], s=20,
                        edgecolors="white", linewidths=0.5, zorder=5,
                    )
        axes[ax_idx].set_title(f"t={t}", fontsize=12)
        axes[ax_idx].axis("off")

    plt.suptitle(f"View {v}: 3D Tracks Reprojected to 2D", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

## Step 9: Evaluation: Computing the TAP-Vid 3D Metrics

Now we evaluate the model's predictions against the **real ground-truth** 3D trajectories that ship with the DexYCB dataset. We use the standard **TAP-Vid** metrics adapted for 3D space.

### Key Metrics Explained

| Metric | What It Measures | Intuition |
|---|---|---|
| **`pts_within_X`** | Positional accuracy (ignoring vis) | Fraction of visible GT points where pred is within X meters of GT |
| **`jaccard_X`** | Combined accuracy + visibility | TP / (TP + FP + FN), combining distance threshold AND visibility |
| **`average_jaccard`** | Average Jaccard across thresholds | **Primary single-number metric** to compare models |
| **`occlusion_accuracy`** | Visibility classification accuracy | How often pred_visible == gt_visible |
| **`survival`** | Long-term tracking robustness | Fraction of the sequence tracked before error exceeds threshold |
| **`ate_visible`** | Average Trajectory Error | Mean Euclidean distance (meters) between pred and GT, for visible pts |
| **`mte_visible`** | Median Trajectory Error | Median distance, more robust to outliers |

### Point Types
The evaluation breaks down metrics by how much each point moves:
- **Static**: Points that barely move (< 1 cm total, e.g., background objects)
- **Dynamic**: Points that move significantly (> 10 cm total, e.g., a person's hand)
- **`dynamic-static-mean`**: Average of the two

In [ ]:
from mvtracker.evaluation.metrics import compute_metrics

# ══════════════════════════════════════════════════════════
# Prepare Ground Truth & Predictions in Metric Space
# ══════════════════════════════════════════════════════════
# Both GT and predictions are in scene-normalized coordinates.
# Multiply by upscale_factor (1/6) to convert to meters.

gt_tracks_meters   = gt_tracks_3d[None] * upscale_factor         # [1, T, N, 3] in meters
pred_tracks_meters = pred_tracks[None] * upscale_factor           # [1, T, N, 3] in meters
query_points_meters = torch.cat([
    query_points[None, :, :1],                                    # timestep (unchanged)
    query_points[None, :, 1:] * upscale_factor,                   # xyz in meters
], dim=-1)                                                        # [1, N, 4]

# GT visibility: visible from ANY view → single (T, N) mask
# compute_metrics expects gt_occluded, so we invert
gt_visible_any = gt_visibility.any(dim=0)                         # [T, N]
gt_occluded = ~gt_visible_any                                     # [T, N], True = occluded
gt_occluded = gt_occluded[None]                                   # [1, T, N]

# Model's predicted occlusion (inverse of visibility boolean)
pred_occluded = ~pred_vis                                         # [T, N]
pred_occluded = pred_occluded[None]                               # [1, T, N]

# ══════════════════════════════════════════════════════════
# Compute the Full Metrics Suite
# ══════════════════════════════════════════════════════════
# Official DexYCB evaluation thresholds (in meters)
distance_thresholds = [0.01, 0.02, 0.05, 0.10, 0.20]  # 1cm – 20cm
survival_threshold  = 0.10                               # 10cm

metrics = compute_metrics(
    query_points=query_points_meters,
    gt_occluded=gt_occluded,
    gt_tracks=gt_tracks_meters,
    pred_occluded=pred_occluded,
    pred_tracks=pred_tracks_meters,
    distance_thresholds=distance_thresholds,
    survival_distance_threshold=survival_threshold,
    query_mode="first",
)

# ══════════════════════════════════════════════════════════
# Display the Results
# ══════════════════════════════════════════════════════════
print("=" * 70)
print("  MVTracker Evaluation on DexYCB Sequence 0")
print("=" * 70)

print(f"\n  Average Jaccard (primary metric):  {metrics['average_jaccard_per_track'].mean().item() * 100:.1f}%")
print(f"  Average Pts Within Threshold:      {metrics['average_pts_within_thresh_per_track'].mean().item() * 100:.1f}%")
print(f"  Occlusion Accuracy:                {metrics['occlusion_accuracy_per_track'].mean().item() * 100:.1f}%")
print(f"  Survival Rate:                     {metrics['survival_per_track'].mean().item() * 100:.1f}%")
print(f"  Average Trajectory Error (ATE):    {metrics['ate_visible_per_track'].mean().item() * 100:.2f} cm")
print(f"  Median Trajectory Error (MTE):     {metrics['mte_visible_per_track'].mean().item() * 100:.2f} cm")

print(f"\n  Per-threshold accuracy:")
for thresh in distance_thresholds:
    key = f"pts_within_{thresh:.2f}_per_track"
    val = metrics[key].mean().item() * 100
    print(f"    Points within {thresh*100:5.1f} cm: {val:.1f}%")

print(f"\n  Per-threshold Jaccard:")
for thresh in distance_thresholds:
    key = f"jaccard_{thresh:.2f}_per_track"
    val = metrics[key].mean().item() * 100
    print(f"    Jaccard @ {thresh*100:5.1f} cm:     {val:.1f}%")

## Step 10: Generate 3D Visualization

Generate a **Rerun `.rrd` file** for interactive 3D visualization. This file contains:
1. The colored 3D point cloud of the scene (from all views).
2. The camera frustums showing each camera's position and orientation.
3. The predicted 3D trajectories as animated lines.

To view the `.rrd` file:
1. Download it from Colab.
2. Go to [app.rerun.io](https://app.rerun.io/version/0.21.0).
3. Drag and drop the file.

In [ ]:
# ══════════════════════════════════════════════════════════
# Generate Rerun .rrd for interactive 3D visualization
# ══════════════════════════════════════════════════════════
import rerun as rr
from mvtracker.utils.visualizer_rerun import log_pointclouds_to_rerun, log_tracks_to_rerun

output_dir = "outputs"
os.makedirs(output_dir, exist_ok=True)

rrd_path = os.path.join(output_dir, "mvtracker_demo.rrd")
rr.init("3dpt", recording_id="v0.16")

print("Logging 3D point clouds to Rerun...")
log_pointclouds_to_rerun(
    dataset_name="demo",
    datapoint_idx=0,
    rgbs=rgbs[None],          # [1, V, T, 3, H, W]
    depths=depths[None],      # [1, V, T, 1, H, W]
    intrs=intrs[None],        # [1, V, T, 3, 3]
    extrs=extrs[None],        # [1, V, T, 3, 4]
    depths_conf=None,
    conf_thrs=[5.0],
    log_only_confident_pc=False,
    radii=-2.45,
    fps=12,
    bbox_crop=None,
    sphere_radius_crop=12.0,
    sphere_center_crop=np.array([0, 0, 0]),
    log_rgb_image=False,
    log_depthmap_as_image_v1=False,
    log_depthmap_as_image_v2=False,
    log_camera_frustrum=True,
    log_rgb_pointcloud=True,
)

print("Logging predicted 3D tracks to Rerun...")
# log_tracks_to_rerun expects batch dim on tensors
log_tracks_to_rerun(
    dataset_name="demo",
    datapoint_idx=0,
    predictor_name="MVTracker",
    gt_trajectories_3d_worldspace=None,
    gt_visibilities_any_view=None,
    query_points_3d=query_points[None],       # [1, N, 4]
    pred_trajectories=pred_tracks[None],      # [1, T, N, 3]
    pred_visibilities=pred_vis[None],         # [1, T, N]
    per_track_results=None,
    radii_scale=1.0,
    fps=12,
    sphere_radius_crop=12.0,
    sphere_center_crop=np.array([0, 0, 0]),
    log_per_interval_results=False,
    max_tracks_to_log=100,
    track_batch_size=50,
    method_id=None,
    color_per_method_id=None,
    memory_lightweight_logging=True,
)

rr.save(rrd_path)
size_mb = os.path.getsize(rrd_path) / 1024 / 1024
print(f"\nSaved Rerun recording to: {os.path.abspath(rrd_path)}  ({size_mb:.1f} MB)")
print("Open this file at https://app.rerun.io/version/0.21.0")

assert os.path.exists(rrd_path), f"Rerun file was not created at {rrd_path}"
print("Rerun export assertion passed.")